In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import gseapy as gp
from IPython.utils import io
from scipy.stats import ttest_ind
from random import shuffle

In [3]:
from sklearn.decomposition import PCA

In [4]:
df = pd.read_csv("data/PBMCs_dataset.csv")
df_t = pd.read_csv("data/tissues_dataset.csv")
df_cl = pd.read_csv("data/cell_lines_dataset.csv")
df_er = pd.read_csv("data/erythropoiesis_dataset.csv")

In [25]:
# RNA data
# PBMCs data
df_imm = pd.pivot(df.loc[df["RNA dataset num."]==1],columns="Cell",index="Gene",values="Log2 TPM")
df_imm = df_imm[df_imm.columns[(df_imm.sum()>0).values]]
df_imm = df_imm.loc[df_imm.sum(axis=1)>0]
# Tissues data
df_tis = pd.pivot(df_t.loc[(df_t["Cell"]+df_t["Gene"]).drop_duplicates().index],columns="Cell",index="Gene",values="Log2 TPM")
df_tis = df_tis[df_tis.columns[(df_tis.sum()>0).values]]
df_tis = df_tis.loc[df_tis.sum(axis=1)>0]
# Cancer cell line data (Proteomic ruler)
df_clt = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==5],columns="Cell",index="Gene",values="Log2 TPM")
df_clt = df_clt[df_clt.columns[(df_clt.sum()>0).values]]
df_clt = df_clt.loc[df_clt.sum(axis=1)>0]
# Cancer cell line data (U2OS)
df_cl2t = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==2],columns="Cell",index="Gene",values="Log2 TPM")
df_cl2t = df_cl2t[df_cl2t.columns[(df_cl2t.sum()>0).values]]
df_cl2t = df_cl2t.loc[df_cl2t.sum(axis=1)>0]
# Cancer cell line data (HeLa)
df_cl3t = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==3],columns="Cell",index="Gene",values="Log2 TPM")
df_cl3t = df_cl3t[df_cl3t.columns[(df_cl3t.sum()>0).values]]
df_cl3t = df_cl3t.loc[df_cl3t.sum(axis=1)>0]
# Cancer cell line data (U-251 MG, A431)
df_cl4t = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==4],columns="Cell",index="Gene",values="Log2 TPM")
df_cl4t = df_cl4t[df_cl4t.columns[(df_cl4t.sum()>0).values]]
df_cl4t = df_cl4t.loc[df_cl4t.sum(axis=1)>0]
# Erythropoiesis data
df_ert = pd.pivot(df_er,columns="Cell",index="Gene",values="Log2 TPM")
df_ert = df_ert[df_ert.columns[(df_ert.sum()>0).values]]
df_ert = df_ert.loc[df_ert.sum(axis=1)>0]
# Put all data and do PCA
df_all = pd.concat([df_imm,df_tis,df_clt,df_cl2t,df_cl3t,df_cl4t,df_ert],axis=1,join="inner")
pca = PCA(n_components=2)
pca.fit(df_all.replace(np.nan,0).transpose().values)
X_pca_imm = pca.transform(df_imm.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_tis = pca.transform(df_tis.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl = pca.transform(df_clt.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl2 = pca.transform(df_cl2t.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl3 = pca.transform(df_cl3t.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl4 = pca.transform(df_cl4t.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_er = pca.transform(df_ert.loc[df_all.index].replace(np.nan,0).transpose().values)

In [12]:
# % variance explained
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
color_sim = [
    (68/255, 138/255, 255/255),
    (0, 150/255, 136/255),
    (139/255, 195/255, 74/255),
    (255/255, 193/255, 7/255),
    (255/255, 152/255, 0),
    (244/255, 67/255, 54/255),
    (173/255, 20/255, 87/255)
]
markers = ["o", "s", "^", "D", "v", "P", "X"]

fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=600)
ax.scatter(X_pca_imm[:, 0], X_pca_imm[:, 1], s=60, color=color_sim[0], marker=markers[0], alpha=1, label="PBMCs")
ax.scatter(X_pca_tis[:, 0], X_pca_tis[:, 1], s=60, color=color_sim[1], marker=markers[1], alpha=1, label="Tissues")
ax.scatter(X_pca_cl[:, 0], X_pca_cl[:, 1], s=60, color=color_sim[2], marker=markers[2], alpha=1, label="Cell lines 1")
ax.scatter(X_pca_cl2[:, 0], X_pca_cl2[:, 1], s=60, color=color_sim[3], marker=markers[3], alpha=1, label="Cell lines 2")
ax.scatter(X_pca_cl3[:, 0], X_pca_cl3[:, 1], s=60, color=color_sim[4], marker=markers[4], alpha=1, label="Cell lines 3")
ax.scatter(X_pca_cl4[:, 0], X_pca_cl4[:, 1], s=60, color=color_sim[5], marker=markers[5], alpha=1, label="Cell lines 4")
ax.scatter(X_pca_er[:, 0], X_pca_er[:, 1], s=60, color=color_sim[6], marker=markers[6], alpha=1, label="Erythropoiesis")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis="x", labelsize=22)
ax.tick_params(axis="y", labelsize=22)
ax.set_xlabel(f"PC1 ({pc1_var:.1f}%)", fontsize=25)
ax.set_ylabel(f"PC2 ({pc2_var:.1f}%)", fontsize=25)
ax.set_title("Bulk RNAseq", fontsize=25)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=18)
plt.savefig("figures/PCA/PCA_BulkRNAseq.pdf", bbox_inches="tight", transparent=True)
plt.close()

In [34]:
# Proteomics data
# PBMCs data
df_imm = pd.pivot(df.loc[df["RNA dataset num."]==1],columns="Cell",index="Gene",values="Log10 Prot. count")
df_imm = df_imm[df_imm.columns[(df_imm.sum()>0).values]]
df_imm = df_imm.loc[df_imm.sum(axis=1)>0]
# Tissues data
df_tis = pd.pivot(df_t.loc[(df_t["Cell"]+df_t["Gene"]).drop_duplicates().index],columns="Cell",index="Gene",values="Log10 Prot. count")
df_tis = df_tis[df_tis.columns[(df_tis.sum()>0).values]]
df_tis = df_tis.loc[df_tis.sum(axis=1)>0]
# Cancer cell line data (Proteomic ruler)
df_clp = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==5],columns="Cell",index="Gene",values="Log10 Prot. count")
df_clp = df_clp[df_clp.columns[(df_clp.sum()>0).values]]
df_clp = df_clp.loc[df_clp.sum(axis=1)>0]
# Cancer cell line data (U2OS)
df_cl2p = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==2],columns="Cell",index="Gene",values="Log10 Prot. count")
df_cl2p = df_cl2p[df_cl2p.columns[(df_cl2p.sum()>0).values]]
df_cl2p = df_cl2p.loc[df_cl2p.sum(axis=1)>0]
# Cancer cell line data (HeLa)
df_cl3p = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==3],columns="Cell",index="Gene",values="Log10 Prot. count")
df_cl3p = df_cl3p[df_cl3p.columns[(df_cl3p.sum()>0).values]]
df_cl3p = df_cl3p.loc[df_cl3p.sum(axis=1)>0]
# Cancer cell line data (U-251 MG, A431)
df_cl4p = pd.pivot(df_cl.loc[df_cl["Prot. count dataset num."]==4],columns="Cell",index="Gene",values="Log10 Prot. count")
df_cl4p = df_cl4p[df_cl4p.columns[(df_cl4p.sum()>0).values]]
df_cl4p = df_cl4p.loc[df_cl4p.sum(axis=1)>0]
# Erythropoiesis data
df_erp = pd.pivot(df_er,columns="Cell",index="Gene",values="Log10 Prot. count")
df_erp = df_erp[df_erp.columns[(df_erp.sum()>0).values]]
df_erp = df_erp.loc[df_erp.sum(axis=1)>0]
# Put all data and do PCA
df_all = pd.concat([df_imm,df_tis,df_clp,df_cl2p,df_cl3p,df_cl4p,df_erp],axis=1,join="inner")
# df_all = df_all/df_all.sum()
pca = PCA(n_components=2)
pca.fit(df_all.replace(np.nan,0).transpose().values)
X_pca_imm = pca.transform(df_imm.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_tis = pca.transform(df_tis.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl = pca.transform(df_clp.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl2 = pca.transform(df_cl2p.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl3 = pca.transform(df_cl3p.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_cl4 = pca.transform(df_cl4p.loc[df_all.index].replace(np.nan,0).transpose().values)
X_pca_er = pca.transform(df_erp.loc[df_all.index].replace(np.nan,0).transpose().values)

In [15]:
# % variance explained
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
color_sim = [
    (68/255, 138/255, 255/255),
    (0, 150/255, 136/255),
    (139/255, 195/255, 74/255),
    (255/255, 193/255, 7/255),
    (255/255, 152/255, 0),
    (244/255, 67/255, 54/255),
    (173/255, 20/255, 87/255)
]
markers = ["o", "s", "^", "D", "v", "P", "X"]

fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=600)
ax.scatter(X_pca_imm[:, 0], X_pca_imm[:, 1], s=60, color=color_sim[0], marker=markers[0], alpha=1, label="PBMCs")
ax.scatter(X_pca_tis[:, 0], X_pca_tis[:, 1], s=60, color=color_sim[1], marker=markers[1], alpha=1, label="Tissues")
ax.scatter(X_pca_cl[:, 0], X_pca_cl[:, 1], s=60, color=color_sim[2], marker=markers[2], alpha=1, label="Cell lines 1")
ax.scatter(X_pca_cl2[:, 0], X_pca_cl2[:, 1], s=60, color=color_sim[3], marker=markers[3], alpha=1, label="Cell lines 2")
ax.scatter(X_pca_cl3[:, 0], X_pca_cl3[:, 1], s=60, color=color_sim[4], marker=markers[4], alpha=1, label="Cell lines 3")
ax.scatter(X_pca_cl4[:, 0], X_pca_cl4[:, 1], s=60, color=color_sim[5], marker=markers[5], alpha=1, label="Cell lines 4")
ax.scatter(X_pca_er[:, 0], X_pca_er[:, 1], s=60, color=color_sim[6], marker=markers[6], alpha=1, label="Erythropoiesis")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis="x", labelsize=22)
ax.tick_params(axis="y", labelsize=22)
ax.set_xlabel(f"PC1 ({pc1_var:.1f}%)", fontsize=25)
ax.set_ylabel(f"PC2 ({pc2_var:.1f}%)", fontsize=25)
ax.set_title("Protein count", fontsize=25)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=18)
plt.savefig("figures/PCA/PCA_prot_count.pdf", bbox_inches="tight", transparent=True)
plt.close()